# DEP-MOUNT 2026 — ML Evidence Visualizations
Run the following cell using your existing 7L-1 and 7L-2 physics-generated DOE CSV files. Figures are synthetic analytical-surrogate diagnostics, not hardware validation.

In [ ]:
"""DEP-MOUNT 2026 — 7L-1 and 7L-2 actual DOE ML visualization.
Run as a single Google Colab cell; requires the two 7L DOE CSVs.
Charts are analytical-surrogate diagnostics, not physical validation.
"""
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error

FILES = [Path('DEP_MOUNT_7L1_Physics_DOE.csv'),
         Path('DEP_MOUNT_7L2_Adapter_Joint_DOE.csv')]
missing_files = [p.name for p in FILES if not p.exists()]
if missing_files:
    try:
        from google.colab import files
        print('Upload the missing DOE CSV files:', missing_files)
        files.upload()
    except ImportError:
        raise FileNotFoundError('Put the two original DOE CSV files in the current working directory.')
if any(not p.exists() for p in FILES):
    raise FileNotFoundError(f'Missing: {[p.name for p in FILES if not p.exists()]}')

out = Path('DEP_MOUNT_ML_Evidence_Figures')
out.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi':105,'savefig.dpi':210,'font.size':10})
made=[]
def publish(name):
    plt.tight_layout()
    p=out/name
    plt.savefig(p,bbox_inches='tight')
    plt.show()
    plt.close()
    made.append(p)

def read_data(p, feature_names, output_names):
    dat=pd.read_csv(p)
    dat.columns=dat.columns.str.strip()
    missing=[c for c in feature_names+output_names if c not in dat.columns]
    if missing: raise ValueError(f'{p.name}: missing columns {missing}; existing columns: {dat.columns.tolist()}')
    sub=dat[feature_names+output_names].apply(pd.to_numeric,errors='coerce').dropna()
    if len(sub)<40: raise ValueError(f'{p.name}: need at least 40 finite DOE cases')
    return dat, sub

def parity(observed,predicted,heading,filename,unit=''):
    x=np.asarray(observed); y=np.asarray(predicted)
    lo=min(x.min(),y.min());hi=max(x.max(),y.max());margin=.04*(hi-lo or 1)
    fig,ax=plt.subplots(figsize=(6.8,5.3))
    ax.scatter(x,y,s=20,alpha=.65)
    ax.plot([lo-margin,hi+margin],[lo-margin,hi+margin],'k--',label='Ideal y = x')
    ax.set(xlabel='Direct analytical model '+unit,ylabel='Held-out ML prediction '+unit,
           title=f'{heading}\nR²={r2_score(x,y):.4f}  MAE={mean_absolute_error(x,y):.4g}  n={len(x)}')
    ax.grid(alpha=.2);ax.legend();publish(filename)

# ------- 7L-1: shell regression --------------------------------------------
f1=['thickness_mm','profile_offset_mm']
t1=['compression_MPa','twist_microrad','shell_mass_g']
raw1,d1=read_data(FILES[0],f1,t1)
X1=d1[f1].to_numpy();Y1=d1[t1].to_numpy()
X1tr,X1te,Y1tr,Y1te=train_test_split(X1,Y1,test_size=.25,random_state=42)
shell=make_pipeline(PolynomialFeatures(degree=3,include_bias=False),StandardScaler(),Ridge(alpha=.01))
shell.fit(X1tr,Y1tr);Y1pred=shell.predict(X1te)
for j,metric,unit in [(0,'Nominal compression','(MPa)'),(1,'Torsional twist','(microrad)'),(2,'Shell mass','(g)')]:
    parity(Y1te[:,j],Y1pred[:,j],f'7L-1 | {metric}',f'7L1_{j+1:02d}_parity.png',unit)

fig,ax=plt.subplots(figsize=(7.4,5.2))
s=ax.scatter(d1['thickness_mm'],d1['shell_mass_g'],c=d1['twist_microrad'],s=18,alpha=.65)
fig.colorbar(s,ax=ax,label='Analytical torsional twist (microrad)')
ax.set(xlabel='Shell thickness (mm)',ylabel='Analytical shell mass proxy (g)',
       title='7L-1 | DOE thickness–mass–twist design space')
ax.grid(alpha=.2);publish('7L1_04_design_space.png')

# ------- 7L-2: adapter regressors -------------------------------------------
f2=['thickness_mm','outer_diameter_mm','inner_diameter_mm','preload_N']
t2=['Adapter mass (g)','Relative flexural rigidity','Thread ratio (5th percentile)',
    'Thread-screen coverage','Friction no-slip coverage','Clamp + depth coverage']
raw2,d2=read_data(FILES[1],f2,t2)
X2=d2[f2].to_numpy();Y2=d2[t2].to_numpy()
X2tr,X2te,Y2tr,Y2te=train_test_split(X2,Y2,test_size=.25,random_state=42)
ridge=make_pipeline(PolynomialFeatures(degree=3,include_bias=False),StandardScaler(),Ridge(alpha=.01))
boost=MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=180,learning_rate=.06,max_leaf_nodes=15,random_state=42))
models={'Polynomial Ridge':ridge,'Gradient Boosting':boost}
preds={};scores=[]
for modelname,model in models.items():
    model.fit(X2tr,Y2tr)
    yhat=model.predict(X2te);preds[modelname]=yhat
    for i,target in enumerate(t2):
        scores.append({'model':modelname,'output':target,'r2':r2_score(Y2te[:,i],yhat[:,i]),
                       'mae':mean_absolute_error(Y2te[:,i],yhat[:,i])})
metrics=pd.DataFrame(scores)
metrics.to_csv(out/'7L2_heldout_metrics_retrained.csv',index=False)
print('\n7L-2 HELD-OUT METRICS (models refitted on original DOE):')
print(metrics.round(5).to_string(index=False))
for i,mname,tag in [(0,'Polynomial Ridge','Mass'),(3,'Gradient Boosting','ThreadCoverage'),(4,'Gradient Boosting','FrictionCoverage')]:
    parity(Y2te[:,i],preds[mname][:,i],f'7L-2 | {tag} | {mname}',
           f'7L2_0{i+5}_{tag}_parity.png')

idx=3
fig,ax=plt.subplots(figsize=(7,5))
ax.scatter(Y2te[:,idx],preds['Gradient Boosting'][:,idx]-Y2te[:,idx],s=22,alpha=.7)
ax.axhline(0,ls='--',color='black')
ax.set(xlabel='Direct analytical thread-screen coverage',ylabel='ML minus analytical coverage',
       title='7L-2 | Held-out residuals: thread coverage')
ax.grid(alpha=.2);publish('7L2_08_thread_residuals.png')

pivot=metrics.pivot(index='output',columns='model',values='r2')
fig,ax=plt.subplots(figsize=(9,5))
xi=np.arange(len(pivot));w=.36
for k,(name,vals) in enumerate(pivot.items()):
    ax.bar(xi+(k-.5)*w,vals,width=w,label=name)
ax.set_xticks(xi,labels=['Mass','Rigidity','Thread ratio','Thread cover.','No-slip cover.','Clamp/depth'])
ax.set(ylabel='Held-out R²',title='7L-2 | Polynomial Ridge versus Gradient Boosting')
ax.axhline(0,color='black',lw=.8);ax.grid(axis='y',alpha=.2);ax.legend()
publish('7L2_09_model_comparison.png')

for idx,name in [(3,'Thread coverage'),(4,'Friction no-slip coverage')]:
    perm=permutation_importance(boost.estimators_[idx],X2te,Y2te[:,idx],scoring='r2',
                                n_repeats=8,random_state=7,n_jobs=1)
    order=np.argsort(perm.importances_mean)
    fig,ax=plt.subplots(figsize=(7.3,4.7))
    ax.barh(np.array(['Thickness','Outer diameter','Inner diameter','Preload'])[order],
            perm.importances_mean[order],xerr=perm.importances_std[order])
    ax.set(xlabel='Held-out R² decrease when feature is shuffled',
           title=f'7L-2 | Permutation sensitivity: {name}\nAssociational, not causal')
    ax.grid(axis='x',alpha=.2)
    publish(f'7L2_{10 if idx==3 else 11}_permutation.png')

# Interpretable *separate* diagnostic tree. Not a depiction of the trained GB model.
tree=DecisionTreeRegressor(max_depth=3,min_samples_leaf=20,random_state=42)
tree.fit(X2tr,Y2tr[:,3])
tree_r2=r2_score(Y2te[:,3],tree.predict(X2te))
fig,ax=plt.subplots(figsize=(18,8))
plot_tree(tree,feature_names=f2,filled=True,rounded=True,fontsize=9,ax=ax,precision=3)
ax.set_title(f'7L-2 | Interpretable diagnostic tree for thread-screen coverage\n'
             f'Independent shallow tree, held-out R²={tree_r2:.3f}; NOT the Gradient Boosting ensemble')
publish('7L2_12_explanatory_decision_tree.png')

# Learning curve on the 7L-2 training partition only: 4-fold CV, thread coverage
lc=HistGradientBoostingRegressor(max_iter=120,learning_rate=.06,max_leaf_nodes=15,random_state=42)
ns,train_scores,val_scores=learning_curve(lc,X2tr,Y2tr[:,3],cv=4,
    train_sizes=np.linspace(.3,1.,5),scoring='neg_mean_absolute_error',shuffle=True,
    random_state=17,n_jobs=1)
fig,ax=plt.subplots(figsize=(7.4,5.0))
for ss,label in [(train_scores,'Train MAE'),(val_scores,'Cross-validation MAE')]:
    v=-ss;ax.plot(ns,v.mean(axis=1),marker='o',label=label)
    ax.fill_between(ns,v.mean(axis=1)-v.std(axis=1),v.mean(axis=1)+v.std(axis=1),alpha=.15)
ax.set(xlabel='Training cases',ylabel='MAE (scenario coverage fraction)',
       title='7L-2 | Thread coverage learning curve (training-only CV)')
ax.grid(alpha=.2);ax.legend();publish('7L2_13_learning_curve.png')

# Engineering trade-off, only data from original DOE; annotated rechecked candidate
valid=(raw2['Geometry/stiffness gate'].astype(float)>.5) if 'Geometry/stiffness gate' in raw2 else pd.Series(True,index=raw2.index)
p=raw2.loc[valid].copy()
base=(518.50,.8167,.3333)
candidate=(410.564,.9722,.1333)
fig,ax=plt.subplots(figsize=(8,5.5))
s=ax.scatter(p['Adapter mass (g)'],p['Thread-screen coverage'],
             c=p['Friction no-slip coverage'],cmap='viridis',s=22,alpha=.65)
fig.colorbar(s,ax=ax,label='Friction no-slip coverage (assumed scenarios)')
ax.scatter(base[0],base[1],marker='s',s=130,label='Current baseline')
ax.scatter(candidate[0],candidate[1],marker='*',s=160,label='Rechecked ML candidate')
ax.axhline(.95,linestyle='--',label='Illustrative 95% coverage target')
ax.set(xlabel='Analytical adapter mass (g)',ylabel='Thread-screen coverage',
       title='7L-2 | Mass–thread–friction trade-off: why low mass is not enough')
ax.grid(alpha=.2);ax.legend(fontsize=9);publish('7L2_14_mass_joint_tradeoff.png')

fig,ax=plt.subplots(figsize=(8,5.2))
ax.scatter(p['preload_N'],p['Friction no-slip coverage'],s=22,alpha=.5,label='DOE cases')
ax.scatter(4000,.3333,s=110,marker='s',label='Current baseline')
ax.scatter(3528.3,.1333,s=160,marker='*',label='Rechecked ML candidate')
ax.axhline(.95,linestyle='--',label='Illustrative 95% target')
ax.set(xlabel='Preload design variable (N/screw)',ylabel='Scenario no-slip coverage',
       title='7L-2 | Preload–friction sensitivity (not measured reliability)')
ax.grid(alpha=.2);ax.legend();publish('7L2_15_preload_friction.png')

# Don't claim feasibility unless every proxy gate actually passes.
req=['Thread-screen coverage','Friction no-slip coverage','Clamp + depth coverage']
full=(p[req].apply(pd.to_numeric,errors='coerce')>=.95).all(axis=1)
print('\n7L-2 DOEs passing all three illustrative 95% coverage gates:',int(full.sum()),'/',len(p))
print('This is a proxy feasibility count, not experimental reliability.')

with zipfile.ZipFile('DEP_MOUNT_7L_ML_Evidence_Figures.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print(f'\nCreated {len(made)} authentic DOE-based charts in {out}/')
print('Created DEP_MOUNT_7L_ML_Evidence_Figures.zip')
print('Evidence limitation: synthetic analytical DOE, no hardware validation.')
